# Install Dependency


In [ ]:
!pip install requests
!pip install beautifulsoup4

Mounted at /content/drive


In [ ]:
import requests
import json
import time
from bs4 import BeautifulSoup
import html

#Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Imports

In [ ]:
!pip install python-dotenv

# Fetch Stack Overflow Data

In [ ]:
def fetch_answer(answer_id):
    url = f"https://api.stackexchange.com/2.3/answers/{answer_id}"
    params = {
        "order": "desc",
        "sort": "votes",
        "site": "stackoverflow",
        "filter": "withbody"
    }

    res = requests.get(url, params=params)
    if res.status_code != 200:
        return None

    data = res.json()
    items = data.get("items", [])
    if len(items) == 0:
        return None

    return items[0].get("body", "")


def clean_html(raw_html):
    if not isinstance(raw_html, str):
        return raw_html

    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator="\n")
    text = html.unescape(text)

    lines = text.splitlines()
    clean_lines = []
    prev_empty = False
    for line in lines:
        if line.strip() == "":
            if not prev_empty:
                clean_lines.append("")
            prev_empty = True
        else:
            clean_lines.append(line)
            prev_empty = False

    return "\n".join(clean_lines).strip()


def summarize_text(text, max_lines=20):
    lines = text.splitlines()
    return "\n".join(lines[:max_lines])


def search_stackoverflow(error_query, pages=1, summary_lines=20):
    results = []

    # define Spring Boot relevant tags & keywords
    spring_tags = ["spring", "spring-boot", "spring-data", "spring-mvc",
                   "spring-security", "spring-jpa", "spring-framework"]
    keywords = ["spring", "spring boot", "spring-boot", "spring data", "spring mvc"]

    for page in range(1, pages + 1):
        print(f"Fetching API page {page}...")
        params = {
            "order": "desc",
            "sort": "votes",
            "pagesize": 20,
            "page": page,
            "q": error_query,
            "site": "stackoverflow",
            "filter": "withbody",
        }

        res = requests.get("https://api.stackexchange.com/2.3/search/advanced", params=params)
        if res.status_code != 200:
            print("API error:", res.status_code, res.text)
            time.sleep(2)
            continue

        data = res.json()
        for item in data.get("items", []):

            # SPRING FILTER
            item_tags = item.get("tags", [])
            title_lower = item.get("title","").lower()
            body_lower  = item.get("body","").lower()
            if not (any(tag in spring_tags for tag in item_tags) or
                    any(k in title_lower for k in keywords) or
                    any(k in body_lower for k in keywords)):
                continue

            question = {
                "id": f"so-{item['question_id']}",
                "source": "StackOverflow",
                "url": item.get("link", ""),
                "title": item.get("title", ""),
                "content": item.get("body", ""),
                "tags": item_tags,
                "solution": "NO_ACCEPTED_ANSWER",
                "chunk_id": f"so-{item['question_id']}-001"
            }

            # fetch accepted answer
            accepted_id = item.get("accepted_answer_id")
            if accepted_id:
                answer = fetch_answer(accepted_id)
                if answer:
                    question["solution"] = answer

            if question["solution"] != "NO_ACCEPTED_ANSWER":

                question["content"] = summarize_text(clean_html(question["content"]), max_lines=summary_lines)
                question["title"]   = clean_html(question["title"])
                question["solution"] = summarize_text(clean_html(question["solution"]), max_lines=summary_lines)

                results.append(question)

        time.sleep(0.5)

    return results


error = "DataIntegrityViolationException"
pages = 2
summary_lines = 20
output_path = "/content/drive/MyDrive/SpringForge/so_results_springboot_12.json"

data = search_stackoverflow(error, pages=pages, summary_lines=summary_lines)


with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Scraping + cleaning + summarizing complete! Saved to:", output_path)
print("Total Spring Boot items:", len(data))


Fetching API page 1...
Fetching API page 2...
Scraping + cleaning + summarizing complete! Saved to: /content/drive/MyDrive/SpringForge/so_results_springboot_12.json
Total Spring Boot items: 22


Normalizing data

In [13]:
import json

with open("/content/drive/MyDrive/SpringForge/springdocs.json") as f:
    spring_docs = json.load(f)

with open("/content/drive/MyDrive/SpringForge/merged_so_data.json") as f:
    so_data = json.load(f)

normalized = []
seen_ids = set()

# Normalize Spring Docs
for item in spring_docs:
    if item["id"] in seen_ids:
        continue
    normalized.append({
        "id": item["id"],
        "source": item["source"],
        "url": item.get("url"),
        "title": item.get("title"),
        "content": item.get("content"),
        "tags": item.get("tags", []),
        "chunk_id": item.get("chunk_id")
    })
    seen_ids.add(item["id"])

# Normalize StackOverflow
for item in so_data:
    if item["id"] in seen_ids:
        continue

    combined_content = item.get("content", "")
    if "solution" in item:
        combined_content += "\n\nSolution:\n" + item["solution"]

    normalized.append({
        "id": item["id"],
        "source": item["source"],
        "url": item.get("url"),
        "title": item.get("title"),
        "content": combined_content,
        "tags": item.get("tags", []),
        "chunk_id": item.get("chunk_id")
    })
    seen_ids.add(item["id"])


with open("/content/drive/MyDrive/SpringForge/normalized_dataset.json", "w") as f:
    json.dump(normalized, f, indent=2)

print("Normalized dataset size:", len(normalized))

Normalized dataset size: 232
